# 토큰 기반 텍스트 분할 (TokenTextSplitter 외)

언어 모델에는 토큰 제한이 있으므로, 청크를 만들 때 **토큰 수**를 기준으로 크기를 재는 것이 좋습니다.
이때는 **실제로 사용할 모델과 같은 토크나이저**로 토큰을 세야 정확합니다.

> **🔄 최신 버전 기준 변경 사항 (langchain-text-splitters 1.x)**
> - tiktoken 인코딩을 명시합니다. `from_tiktoken_encoder()` 와 `TokenTextSplitter` 의 기본 인코딩은 여전히 **`gpt2`** 이므로,
>   GPT-4o 이후 OpenAI 모델이 쓰는 **`o200k_base`** 를 직접 지정해야 실제 토큰 수와 맞습니다.
> - `TokenTextSplitter` 는 한글처럼 한 글자가 여러 토큰인 언어에서 **글자가 깨질 수 있습니다**. 공식 문서도 이 경우
>   `RecursiveCharacterTextSplitter.from_tiktoken_encoder()` 사용을 권장하므로, 깨짐 여부를 확인하는 코드와 대안을 추가했습니다.
> - spaCy: 영어 모델(`en_core_web_sm`) 대신 **한국어 파이프라인 `ko_core_news_sm`** 을 사용합니다.
> - SentenceTransformers: 원본의 `chunk_size=200` 은 이 분할기에서 **실제 청크 크기를 결정하지 않습니다**. 올바른 인자인 `tokens_per_chunk` 로 수정하고, 한국어를 지원하는 다국어 모델을 지정했습니다.
> - NLTK: 최신 NLTK는 `punkt` 대신 **`punkt_tab`** 리소스를 사용합니다.
> - KoNLPy: 불필요한 `import chunk` 를 제거했습니다. (Java 설치 필요)
> - Hugging Face: `GPT2TokenizerFast` 대신 **`AutoTokenizer`** 를 사용하고, 한국어를 잘 다루는 모델의 토크나이저로 바꿨습니다.

In [ ]:
%pip install -qU langchain-text-splitters tiktoken

## tiktoken

`tiktoken` 은 OpenAI가 만든 빠른 BPE 토크나이저입니다.

| 인코딩 | 사용 모델 |
|---|---|
| `o200k_base` | GPT-4o, GPT-4.1, o-시리즈 등 최신 OpenAI 모델 |
| `cl100k_base` | GPT-4, GPT-3.5-turbo, text-embedding-3-* |
| `gpt2` | LangChain 분할기의 **기본값** (구형) |

샘플 텍스트를 읽습니다.

In [ ]:
from pathlib import Path

file = Path("./data/appendix-keywords.txt").read_text(encoding="utf-8")
print(file[:350])

### CharacterTextSplitter.from_tiktoken_encoder

- 🔄 `encoding_name="o200k_base"` 를 명시합니다. (또는 `model_name="gpt-4o"` 처럼 모델명으로 지정할 수도 있습니다.)
- 텍스트는 `CharacterTextSplitter` 의 구분자(`"\n\n"`)로만 나뉘고, tiktoken은 **병합 시 크기 측정**에만 사용됩니다.
  따라서 하나의 조각이 크면 청크가 `chunk_size` 토큰을 **넘을 수 있습니다**.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="o200k_base",  # 🔄 최신 OpenAI 모델용 인코딩
    chunk_size=300,
    chunk_overlap=0,
)
texts = text_splitter.split_text(file)
print(len(texts))

In [ ]:
print(texts[0])

🔄 실제 토큰 수를 tiktoken으로 직접 세어 확인해 봅니다.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("o200k_base")
token_counts = [len(enc.encode(t)) for t in texts]
print("청크별 토큰 수(앞 10개):", token_counts[:10])
print("최대 토큰 수:", max(token_counts))

### RecursiveCharacterTextSplitter.from_tiktoken_encoder (권장)

크기를 **엄격하게** 지켜야 한다면 재귀 분할기를 사용합니다. 조각이 크면 더 작은 구분자로 다시 나누므로 모든 청크가 `chunk_size` 토큰 이하가 됩니다.
또한 글자 경계에서만 자르기 때문에 한글이 깨지지 않습니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="o200k_base",
    chunk_size=300,
    chunk_overlap=0,
)
recursive_texts = recursive_splitter.split_text(file)
print("최대 토큰 수:", max(len(enc.encode(t)) for t in recursive_texts))

## TokenTextSplitter

tiktoken으로 텍스트를 **토큰 단위로 직접** 자릅니다. 각 청크가 `chunk_size` 토큰 이하임이 보장되지만,
한글처럼 한 글자가 2개 이상의 토큰으로 인코딩되는 경우 **글자 중간에서 잘려** 깨진 문자(`\ufffd`)가 생길 수 있습니다.

In [ ]:
from langchain_text_splitters import TokenTextSplitter

text_splitter = TokenTextSplitter(
    encoding_name="o200k_base",  # 🔄 기본값 gpt2 대신 명시
    chunk_size=300,
    chunk_overlap=0,
)

texts = text_splitter.split_text(file)
print(texts[0])

In [ ]:
# 🔄 깨진 문자(유니코드 대체 문자 U+FFFD)가 포함된 청크 수를 확인합니다.
broken = [i for i, t in enumerate(texts) if "\ufffd" in t]
print(f"전체 {len(texts)}개 중 깨진 문자가 있는 청크: {len(broken)}개 {broken[:10]}")

> 한국어 문서라면 `TokenTextSplitter` 보다 위의 `RecursiveCharacterTextSplitter.from_tiktoken_encoder()` 를 사용하는 것이 안전합니다.

## spaCy

spaCy는 Python/Cython으로 작성된 고급 자연어 처리 라이브러리입니다.

1. 텍스트 분할 방식: **spaCy** 문장 분리기
2. 청크 크기 측정 방식: **문자 수**

🔄 `SpacyTextSplitter` 의 기본 파이프라인은 영어 모델 `en_core_web_sm` 입니다. 샘플이 한국어이므로 **한국어 파이프라인 `ko_core_news_sm`** 을 사용합니다.
모델 다운로드 없이 규칙 기반으로만 문장을 나누고 싶다면 `pipeline="sentencizer"` 를 지정할 수도 있습니다.

In [ ]:
%pip install -qU spacy

In [ ]:
# 🔄 `!python -m spacy download` 는 노트북 커널과 다른 파이썬을 가리킬 수 있으므로,
# 현재 커널에서 직접 다운로드 함수를 호출합니다.
import spacy.cli

spacy.cli.download("ko_core_news_sm")

In [ ]:
from pathlib import Path

file = Path("./data/appendix-keywords.txt").read_text(encoding="utf-8")
print(file[:350])

In [ ]:
from langchain_text_splitters import SpacyTextSplitter

text_splitter = SpacyTextSplitter(
    pipeline="ko_core_news_sm",  # 🔄 한국어 파이프라인
    chunk_size=200,
    chunk_overlap=50,
)

In [ ]:
texts = text_splitter.split_text(file)
print(texts[0])

## SentenceTransformers

`SentenceTransformersTokenTextSplitter` 는 sentence-transformers 임베딩 모델의 토크나이저로 토큰을 세어,
해당 모델의 입력 길이에 맞게 청크를 만듭니다.

- 🔄 청크 크기는 **`tokens_per_chunk`** 로 지정합니다. (원본의 `chunk_size` 는 이 분할기의 청크 크기에 반영되지 않습니다.)
- `tokens_per_chunk` 는 모델의 최대 입력 길이를 넘을 수 없습니다. 생략하면 모델 최대 길이가 사용됩니다.
- 🔄 기본 모델 `all-mpnet-base-v2` 는 영어 모델이므로, 한국어를 지원하는 **`intfloat/multilingual-e5-small`**(최대 512 토큰)을 지정했습니다.
  실제로는 **벡터DB에 넣을 때 사용할 임베딩 모델과 같은 모델**을 지정하세요.

In [ ]:
%pip install -qU sentence-transformers

In [ ]:
from langchain_text_splitters import SentenceTransformersTokenTextSplitter

splitter = SentenceTransformersTokenTextSplitter(
    model_name="intfloat/multilingual-e5-small",  # 🔄 다국어 모델
    tokens_per_chunk=200,                          # 🔄 chunk_size → tokens_per_chunk
    chunk_overlap=0,
)
print("청크당 최대 토큰 수:", splitter.maximum_tokens_per_chunk)

In [ ]:
from pathlib import Path

file = Path("./data/appendix-keywords.txt").read_text(encoding="utf-8")
print(file[:350])

`file` 의 토큰 수를 셉니다. `count_tokens()` 결과에는 시작/종료 특수 토큰 2개가 포함되므로 빼 줍니다.

In [ ]:
count_start_and_stop_tokens = 2

text_token_count = splitter.count_tokens(text=file) - count_start_and_stop_tokens
print(text_token_count)

In [ ]:
text_chunks = splitter.split_text(text=file)
print(len(text_chunks))

In [ ]:
# 두 번째 청크(인덱스 1)를 출력합니다.
print(text_chunks[1])

## NLTK

NLTK(Natural Language Toolkit)는 영어 자연어 처리를 위한 파이썬 라이브러리입니다.

1. 텍스트 분할 방식: NLTK 문장 토크나이저
2. 청크 크기 측정 방식: 문자 수

🔄 NLTK 3.9 이후 문장 토크나이저는 `punkt` 대신 **`punkt_tab`** 리소스를 사용합니다. `punkt` 만 받으면 `LookupError` 가 발생합니다.

> NLTK의 문장 분리 모델은 영어(유럽어) 중심입니다. 한국어 문서라면 뒤에 나오는 KoNLPy나 앞의 spaCy 한국어 파이프라인이 더 적합합니다.

In [ ]:
%pip install -qU nltk

In [ ]:
import nltk

nltk.download("punkt_tab")  # 🔄 punkt → punkt_tab

In [ ]:
from pathlib import Path

file = Path("./data/appendix-keywords.txt").read_text(encoding="utf-8")
print(file[:350])

In [ ]:
from langchain_text_splitters import NLTKTextSplitter

text_splitter = NLTKTextSplitter(
    chunk_size=200,
    chunk_overlap=0,
)

In [ ]:
texts = text_splitter.split_text(file)
print(texts[0])

## KoNLPy

KoNLPy(Korean NLP in Python)는 한국어 자연어 처리 패키지입니다.
영어용 토크나이저는 한국어의 고유한 형태·의미 구조를 이해하지 못하므로 한국어 처리에 효과적이지 않습니다.

### Kkma 분석기를 사용한 한국어 문장 분할

`KonlpyTextSplitter` 는 KoNLPy의 `Kkma`(Korean Knowledge Morpheme Analyzer)로 문장을 분리합니다.
`Kkma` 는 분석이 상세한 대신 **느리므로**, 속도보다 분석 깊이가 중요한 경우에 적합합니다.

> ⚠️ KoNLPy는 내부적으로 JVM을 사용하므로 **Java(JDK 8 이상)** 가 설치되어 있고 `JAVA_HOME` 이 설정되어 있어야 합니다.

In [ ]:
%pip install -qU konlpy

In [ ]:
from pathlib import Path

file = Path("./data/appendix-keywords.txt").read_text(encoding="utf-8")
print(file[:350])

In [ ]:
from langchain_text_splitters import KonlpyTextSplitter

text_splitter = KonlpyTextSplitter(chunk_size=200, chunk_overlap=50)

In [ ]:
texts = text_splitter.split_text(file)
print(texts[0])

## Hugging Face 토크나이저

Hugging Face 토크나이저로 토큰 수를 세어 청크 크기를 측정할 수 있습니다.

- 분할 방식: 전달된 구분자 단위
- 크기 측정: Hugging Face 토크나이저의 토큰 수

🔄 변경점
- `GPT2TokenizerFast` 처럼 모델별 클래스를 직접 쓰기보다 **`AutoTokenizer.from_pretrained()`** 를 쓰는 것이 현재 권장 방식입니다.
- GPT-2 토크나이저는 한국어를 매우 비효율적으로 토큰화합니다. 여기서는 한국어 RAG에서 많이 쓰는 다국어 임베딩 모델 **`BAAI/bge-m3`** 의 토크나이저를 사용합니다.
  (실무에서는 **실제로 사용할 임베딩/LLM 모델의 토크나이저**를 지정하세요. 토크나이저 파일만 내려받으므로 모델 가중치는 받지 않습니다.)

In [ ]:
%pip install -qU transformers

In [ ]:
from transformers import AutoTokenizer

hf_tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

In [ ]:
from pathlib import Path

file = Path("./data/appendix-keywords.txt").read_text(encoding="utf-8")
print(file[:350])

`from_huggingface_tokenizer()` 로 분할기를 만듭니다. 크기 상한을 엄격히 지키려면 `RecursiveCharacterTextSplitter` 버전을 사용합니다.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

# 구분자("\n\n")로만 나누고 토큰 수로 병합 → 300 토큰을 넘는 청크가 생길 수 있음
text_splitter = CharacterTextSplitter.from_huggingface_tokenizer(
    hf_tokenizer,
    chunk_size=300,
    chunk_overlap=50,
)
texts = text_splitter.split_text(file)

# 재귀적으로 나눔 → 모든 청크가 300 토큰 이하
recursive_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    hf_tokenizer,
    chunk_size=300,
    chunk_overlap=50,
)
recursive_texts = recursive_splitter.split_text(file)

In [ ]:
print(texts[1])

In [ ]:
# 🔄 두 방식의 최대 토큰 수를 비교합니다. (특수 토큰은 제외하고 셉니다)
def count(t: str) -> int:
    return len(hf_tokenizer.encode(t, add_special_tokens=False))

print("Character  최대 토큰 수:", max(count(t) for t in texts))
print("Recursive  최대 토큰 수:", max(count(t) for t in recursive_texts))